In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


import numpy as np
import pandas as pd

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

2023-02-10 12:55:47.956838: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 12:55:48.066187: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-02-10 12:55:48.557018: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-02-10 12:55:48.557063: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

In [2]:
col = ["duration","protocol_type","service","flag","src-bytes","dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins","logged_in","num-compromised","root-shell","su-attempted","num_root","num_file_creation","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_error_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","class_attack","class_score"]
url = "KDDTest+1.csv"
data = pd.read_csv(url,names=col)
data.shape

(22544, 43)

In [3]:
# remove attribute 'difficulty_level'
data.drop(['class_score'],axis=1,inplace=True)
data.shape

(22544, 42)

In [4]:
# changing attack labels to their respective attack class
# total 38 different sub-attacks 
def change_label(df):
  df.class_attack.replace(['apache2','back','land','neptune','mailbomb','pod','processtable','smurf','teardrop','udpstorm','worm'],'Dos',inplace=True)
  df.class_attack.replace(['ftp_write','guess_passwd','httptunnel','imap','multihop','named','phf','sendmail',
       'snmpgetattack','snmpguess','spy','warezclient','warezmaster','xlock','xsnoop'],'R2L',inplace=True)
  df.class_attack.replace(['ipsweep','mscan','nmap','portsweep','saint','satan'],'Probe',inplace=True)
  df.class_attack.replace(['buffer_overflow','loadmodule','perl','ps','rootkit','sqlattack','xterm'],'U2R',inplace=True)


In [5]:
# calling change_label() function
change_label(data)

In [6]:
# distribution of attack classes
data.class_attack.value_counts()

normal    9711
Dos       7460
R2L       2885
Probe     2421
U2R         67
Name: class_attack, dtype: int64

In [7]:
#### data normalization #####

In [8]:
# importing required libraries for normalizing data
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

In [9]:
# selecting numeric attributes columns from data
numeric_col = data.select_dtypes(include='number').columns

# using standard scaler for normalizing
std_scaler = StandardScaler()
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df


In [10]:
# calling the normalization() function
data = normalization(data.copy(),numeric_col)

# data after normalization
data.head()

,duration,protocol_type,service,flag,src-bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class_attack
0,-0.155534,tcp,private,REJ,-0.021988,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-1.169697,-1.305370,-0.138370,-0.431856,-0.229980,-0.358118,-0.35275,1.979791,1.929116,Dos
1,-0.155534,tcp,private,REJ,-0.021988,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-1.250212,-1.397181,-0.138370,-0.431856,-0.229980,-0.358118,-0.35275,1.979791,1.929116,Dos
2,-0.154113,tcp,ftp_data,SF,0.005473,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.489800,0.002934,-0.228985,1.559906,0.004234,-0.358118,-0.35275,-0.602719,-0.565483,normal
3,-0.155534,icmp,eco_i,SF,-0.021946,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.749234,0.898090,-0.410217,2.833328,3.049016,-0.358118,-0.35275,-0.602719,-0.565483,Probe
4,-0.154823,tcp,telnet,RSTO,-0.021988,-0.096189,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.489800,-0.685647,0.360018,-0.333901,0.004234,-0.358118,-0.35275,1.540764,1.205682,Probe


In [11]:
#### one hot encoding ####

In [12]:
# selecting categorical data attributes
cat_col = ['protocol_type','service','flag']

# creating a dataframe with only categorical attributes
categorical = data[cat_col]
categorical.head()

,protocol_type,service,flag
0,tcp,private,REJ
1,tcp,private,REJ
2,tcp,ftp_data,SF
3,icmp,eco_i,SF
4,tcp,telnet,RSTO


In [13]:
# one-hot-encoding categorical attributes using pandas.get_dummies() function
categorical = pd.get_dummies(categorical,columns=cat_col)
categorical.head()

,protocol_type_icmp,protocol_type_tcp,protocol_type_udp,service_IRC,service_X11,service_Z39_50,service_auth,service_bgp,service_courier,service_csnet_ns,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [14]:
### multiclass classification #####

In [15]:
# creating a dataframe with multi-class labels (Dos,Probe,R2L,U2R,normal)
multi_data_test = data.copy()
multi_label_test = pd.DataFrame(multi_data_test.class_attack)

In [16]:
# label encoding (0,1,2,3,4) multi-class labels (Dos,normal,Probe,R2L,U2R)
le2_test = preprocessing.LabelEncoder()
enc_label = multi_label_test.apply(le2_test.fit_transform)
multi_data_test['intrusion'] = enc_label

In [17]:
le2_test.classes_

array(['Dos', 'Probe', 'R2L', 'U2R', 'normal'], dtype=object)

In [18]:
# one-hot-encoding attack label
multi_data_test = pd.get_dummies(multi_data_test,columns=['class_attack'],prefix="",prefix_sep="") 
multi_data_test['class_attack'] = multi_label_test
multi_data_test

,duration,protocol_type,service,flag,src-bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,intrusion,Dos,Probe,R2L,U2R,normal,class_attack
0,-0.155534,tcp,private,REJ,-0.021988,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,1.979791,1.929116,0,1,0,0,0,0,Dos
1,-0.155534,tcp,private,REJ,-0.021988,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,1.979791,1.929116,0,1,0,0,0,0,Dos
2,-0.154113,tcp,ftp_data,SF,0.005473,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,-0.602719,-0.565483,4,0,0,0,0,1,normal
3,-0.155534,icmp,eco_i,SF,-0.021946,-0.096896,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,-0.602719,-0.565483,1,0,1,0,0,0,Probe
4,-0.154823,tcp,telnet,RSTO,-0.021988,-0.096189,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,1.540764,1.205682,1,0,1,0,0,0,Probe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,-0.155534,tcp,smtp,SF,-0.020309,-0.081202,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,-0.602719,-0.565483,4,0,0,0,0,1,normal
22540,-0.155534,tcp,http,SF,-0.021318,-0.052690,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,-0.602719,-0.565483,4,0,0,0,0,1,normal
22541,-0.155534,tcp,http,SF,0.093373,0.294926,-0.017624,-0.059104,-0.019459,2.040705,...,-0.35275,-0.421943,-0.390861,0,1,0,0,0,0,Dos
22542,-0.155534,udp,domain_u,SF,-0.021899,-0.094917,-0.017624,-0.059104,-0.019459,-0.113521,...,-0.35275,-0.602719,-0.565483,4,0,0,0,0,1,normal


In [19]:
#### feature extraction ####

In [20]:
# creating a dataframe with only numeric attributes of multi-class dataset and encoded label attribute 
numeric_multi = multi_data_test[numeric_col]
numeric_multi['intrusion'] = multi_data_test['intrusion']

/tmp/ipykernel_17695/3380547693.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_multi['intrusion'] = multi_data_test['intrusion']


In [21]:
# finding the attributes which have more than 0.5 correlation with encoded attack label attribute 
corr = numeric_multi.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.5]
highest_corr.sort_values(ascending=True)

dst_host_srv_rerror_rate    0.540813
srv_error_rate              0.547787
rerror_rate                 0.553599
dst_host_rerror_rate        0.570808
logged_in                   0.572717
dst_host_srv_count          0.575061
dst_host_same_srv_rate      0.605472
same_srv_rate               0.614762
intrusion                   1.000000
Name: intrusion, dtype: float64

In [22]:
# selecting attributes found by using pearson correlation coefficient
numeric_multi = multi_data_test[['logged_in','srv_serror_rate','rerror_rate','dst_host_rerror_rate',
                        'dst_host_same_srv_rate','dst_host_srv_serror_rate','dst_host_srv_count','same_srv_rate']]

# joining the selected attribute with the one-hot-encoded categorical dataframe
numeric_multi = numeric_multi.join(categorical)
# then joining encoded, one-hot-encoded, and original attack label attribute
multi_data_test = numeric_multi.join(multi_data_test[['intrusion','Dos','Probe','R2L','U2R','normal','class_attack']])

In [23]:
# saving final dataset to disk
#multi_data_test.to_csv('datasets/multi_data_test.csv')

# final dataset for multi-class classification
multi_data_test

,logged_in,srv_serror_rate,rerror_rate,dst_host_rerror_rate,dst_host_same_srv_rate,dst_host_srv_serror_rate,dst_host_srv_count,same_srv_rate,protocol_type_icmp,protocol_type_tcp,...,flag_S3,flag_SF,flag_SH,intrusion,Dos,Probe,R2L,U2R,normal,class_attack
0,-0.890373,-0.347390,1.830141,1.979791,-1.305370,-0.35275,-1.169697,-1.697859,0,1,...,0,0,0,0,1,0,0,0,0,Dos
1,-0.890373,-0.347390,1.830141,1.979791,-1.397181,-0.35275,-1.250212,-1.770589,0,1,...,0,0,0,0,1,0,0,0,0,Dos
2,-0.890373,-0.347390,-0.573079,-0.602719,0.002934,-0.35275,-0.489800,0.629488,0,1,...,0,1,0,4,0,0,0,0,1,normal
3,-0.890373,-0.347390,-0.573079,-0.602719,0.898090,-0.35275,-0.749234,0.629488,1,0,...,0,1,0,1,0,1,0,0,0,Probe
4,-0.890373,0.054856,1.830141,1.540764,-0.685647,-0.35275,-0.489800,0.629488,0,1,...,0,0,0,1,0,1,0,0,0,Probe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22539,1.123125,-0.347390,-0.573079,-0.602719,0.255414,-0.35275,0.002232,0.629488,0,1,...,0,1,0,4,0,0,0,0,1,normal
22540,1.123125,-0.347390,-0.573079,-0.602719,0.898090,-0.35275,1.022079,0.629488,0,1,...,0,1,0,4,0,0,0,0,1,normal
22541,1.123125,-0.347390,-0.573079,-0.421943,0.898090,-0.35275,1.022079,0.629488,0,1,...,0,1,0,0,1,0,0,0,0,Dos
22542,-0.890373,-0.347390,-0.573079,-0.602719,0.875137,-0.35275,0.995240,0.629488,0,0,...,0,1,0,4,0,0,0,0,1,normal


In [24]:
import warnings

warnings.filterwarnings("ignore")

X = multi_data_test.iloc[:,0:86] # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = multi_data_test['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)

In [25]:
X_train = preprocessing.scale(X_train)
X_train = preprocessing.normalize(X_train)

In [26]:
X_validation = preprocessing.scale(X_validation)
X_validation = preprocessing.normalize(X_validation)

In [27]:
print(len(X_train), "Training sequences",X_train.shape)
print(len(X_validation), "Validation sequences",X_validation.shape)

18035 Training sequences (18035, 86)
4509 Validation sequences (4509, 86)


In [28]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))
X_validation = np.reshape(X_validation,(X_validation.shape[0],X_validation.shape[1],1))

In [29]:
import time

Model = keras.Sequential([

        keras.layers.Conv2D(96,(4,4),input_shape=(X_train.shape[1],X_train.shape[2],1),activation='relu',padding='same'),
        keras.layers.Conv2D(64,(3,3),activation="relu",padding='same'),
        keras.layers.Conv2D(32,(2,2),activation="relu",padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512,activation="relu"),
        keras.layers.Dense(128,activation="relu"),
        keras.layers.Dense(32,activation="relu"),
        keras.layers.Dense(5,activation="softmax"),
    
    
    ])

Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
start_time = time.time()
#Training the model
Model.fit(X_train, Y_train, epochs=5, batch_size=64) 
Model.summary()

# Final evaluation of the model
scores = Model.evaluate(X_validation, Y_validation, verbose=0)
delta = time.time()- start_time
print("Accuracy: %.2f%%" % (scores[1]*100))
print("Training time: %.2f sec" % (delta))

2023-02-10 12:55:49.981208: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 12:55:51.165398: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14769 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:19:00.0, compute capability: 7.5
2023-02-10 12:55:51.166039: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14769 MB memory:  -> device: 1, name: Quadro RTX 5000, pci bus id: 0000:1a:00.0, compute capability: 7.5
2023-02-10 12:55:51.166405: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/tas

Epoch 1/5


2023-02-10 12:55:52.722476: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8600
2023-02-10 12:55:53.294699: I tensorflow/compiler/xla/service/service.cc:173] XLA service 0x7f2d76333720 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2023-02-10 12:55:53.294721: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (0): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 12:55:53.294725: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (1): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 12:55:53.294728: I tensorflow/compiler/xla/service/service.cc:181]   StreamExecutor device (2): Quadro RTX 5000, Compute Capability 7.5
2023-02-10 12:55:53.298287: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2023-02-10 12:55:53.382729: I tensorflow/compiler/jit/xla_compi

282/282 [==============================] - 6s 9ms/step - loss: 0.4032 - sparse_categorical_accuracy: 0.8578
Epoch 2/5
282/282 [==============================] - 2s 9ms/step - loss: 0.2205 - sparse_categorical_accuracy: 0.9196
Epoch 3/5
282/282 [==============================] - 3s 9ms/step - loss: 0.1756 - sparse_categorical_accuracy: 0.9306
Epoch 4/5
282/282 [==============================] - 3s 9ms/step - loss: 0.1550 - sparse_categorical_accuracy: 0.9370
Epoch 5/5
282/282 [==============================] - 3s 9ms/step - loss: 0.1503 - sparse_categorical_accuracy: 0.9378
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 86, 1, 96)         1632      
                                                                 
 conv2d_1 (Conv2D)           (None, 86, 1, 64)         55360     
                                                                 
 con

In [30]:
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error
                             ,mean_absolute_error,roc_auc_score,confusion_matrix)
from sklearn import metrics

In [31]:
y_pred = Model.predict(X_validation, verbose = 0)
# predict crisp classes for test set
yhat_classes = np.argmax(y_pred,axis=1)
# reduce to 1d array
yhat_probs = y_pred[:, 0]

 
# accuracy: (tp + tn) / (p + n)
accuracy = accuracy_score(Y_validation, yhat_classes)
print('Accuracy: %f' % accuracy)
# precision tp / (tp + fp)
precision = precision_score(Y_validation, yhat_classes,average='macro')
print('Precision: %f' % precision)
# recall: tp / (tp + fn)
recall = recall_score(Y_validation, yhat_classes,average='macro')
print('Recall: %f' % recall)
# f1: 2 tp / (2 tp + fp + fn)
f1 = f1_score(Y_validation, yhat_classes,average='macro')
print('F1 score: %f' % f1)


# ROC AUC
#auc = roc_auc_score(Y_validation, yhat_probs,multi_class='ovr')
#print('ROC AUC: %f' % auc)
# confusion matrix
matrix = confusion_matrix(Y_validation, yhat_classes)
print(matrix)
# false alaram rate
false_alaram_rate = matrix[1,0]/(matrix[1,0]+matrix[0,0])
print('false alaram rate: %f' % false_alaram_rate)

Accuracy: 0.937902
Precision: 0.950484
Recall: 0.767926
F1 score: 0.804533
[[1475    9    1    0   37]
 [  35  429    0    0    4]
 [   2    3  443    0  114]
 [   0    0    3    2    5]
 [  22   17   28    0 1880]]
false alaram rate: 0.023179
